# CineMatch: MovieLens 100K exploration
Run `python -m scripts.download_data` from the repository root first. This notebook inspects density, rating distributions, genre imbalance and time boundaries; it does not tune on the test data.

In [ ]:
from pathlib import Path
import sys
root = Path.cwd() if (Path.cwd() / 'src').exists() else Path.cwd().parent
sys.path.insert(0, str(root))
from src.data import load_movielens, temporal_split, GENRES  # noqa: E402
movies, ratings = load_movielens(root / 'data' / 'ml-100k')
ratings.head()

In [ ]:
import pandas as pd
from src.catalog import latest_ratings
distinct = latest_ratings(ratings)
pd.Series({'users': ratings.user_id.nunique(), 'movies': len(movies), 'source_events': len(ratings), 'distinct_user_title_pairs': len(distinct), 'density': len(distinct) / (ratings.user_id.nunique() * len(movies)), 'mean_rating': ratings.rating.mean()})

In [ ]:
import matplotlib.pyplot as plt
ratings.rating.value_counts().sort_index().plot.bar(title='Rating distribution')
plt.show()
movies[GENRES].sum().sort_values().plot.barh(title='Genre distribution (multi-label)')
plt.show()

In [ ]:
distinct.groupby('movie_id').size().describe(percentiles=[.1, .5, .9, .99])

In [ ]:
train, validation, test = temporal_split(ratings)
pd.DataFrame([{'split': name, 'ratings': len(frame), 'first': pd.to_datetime(frame.timestamp.min(), unit='s', utc=True), 'last': pd.to_datetime(frame.timestamp.max(), unit='s', utc=True), 'users': frame.user_id.nunique()} for name, frame in [('train', train), ('validation', validation), ('test', test)]])

## Interpretation
The user/movie matrix is sparse; a missing rating must not be treated as zero. The 1,682 source rows become 1,664 canonical film identities. Preserve all 100,000 rating events before temporal splitting; use the latest available canonical user/title pair only inside a training boundary. Density uses 99,693 distinct full-data pairs. Genres are imbalanced and multi-label. Timestamp ties never cross split boundaries. Read `docs/methodology.md` for protocol 2, unseen-user onboarding and uncertainty. Run `python -m scripts.eda` for the saved overview figure. Series metadata are not part of the interaction benchmark.
